# Каузальный пайплайн: каузальные модели vs ML при наличии вмешательства

**Цель:** на синтетических данных с известной причинно-следственной структурой:
1. **STEP 1** — Определить DAG (причины, конфаундеры, инструменты)
2. **STEP 2** — Отобрать контрольные переменные (экспертный + эвристический)
3. **STEP 3** — Каузальные модели: OLS+backdoor, DML, VAR+Granger
4. **STEP 4** — ML-базлайны: ARIMA, Ridge (наивный, без C), RandomForest, LightGBM
5. **STEP 5** — Сравнение MSE, восстановление θ*, поведение при вмешательстве

**Данные:** Сценарий 1 — линейный VAR(1) с конфаундером C:
$$C_t = 0.6 C_{t-1} + \varepsilon_C \qquad X_t = 0.3 X_{t-1} + 0.4 C_{t-1} + \varepsilon_X$$
$$Y_t = 0.2 Y_{t-1} + \underbrace{0.8}_{\theta^*} X_{t-1} - 0.3 C_{t-1} + \varepsilon_Y$$

Вмешательство: $X_{t_0} \mathrel{+}= \Delta = +2\sigma_X$ при $t_0 = 300$.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb

SEED = 42; TRUE_EFFECT = 0.8
T = 600; T0 = 300; TRAIN_END = 450; N_REAL = 10
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.titlesize':12,'lines.linewidth':1.5})
print(f'Ready. T={T} | t0={T0} | train=[0..{TRAIN_END}) | test=[{TRAIN_END}..{T})')

---
## STEP 1 — DAG Definition

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class DAGSpec:
    target:      str
    causes:      List[str]
    confounders: List[str]
    mediators:   List[str]
    instruments: List[str]
    true_coef:   Dict[str,float]
    notes:       str = ''

DAG = DAGSpec(
    target      = 'Y',
    causes      = ['X'],
    confounders = ['C'],
    mediators   = [],
    instruments = [],
    true_coef   = {'Y_lag1':0.2, 'X_lag1':0.8, 'C_lag1':-0.3},
    notes       = 'C влияет и на X, и на Y (backdoor X←C→Y). Без контроля C оценка θ смещена.'
)
print('DAG:')
print(f'  Цель        : {DAG.target}')
print(f'  Причины     : {DAG.causes}')
print(f'  Конфаундеры : {DAG.confounders}')
print(f'  θ*(X_lag1)  : {DAG.true_coef["X_lag1"]}')
print(f'  Примечание  : {DAG.notes}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.set_xlim(0,10); ax.set_ylim(0,5); ax.axis('off')
ax.set_title('STEP 1 — DAG: линейный VAR(1) с конфаундером', fontweight='bold')
nodes = {'C':(5.0,4.2,'#a8e6a3'),'X':(1.5,1.5,'#f9c49a'),'Y':(8.5,1.5,'#9ecae1')}
for name,(x,y,col) in nodes.items():
    ax.add_patch(plt.Circle((x,y),0.72,color=col,ec='black',lw=2,zorder=3))
    ax.text(x,y,name,ha='center',va='center',fontsize=16,fontweight='bold',zorder=4)
for s,d,lbl,col in [('C','X','0.4','#2ca02c'),('C','Y','-0.3','#2ca02c'),('X','Y','θ*=0.8','#d62728')]:
    sx,sy,_=nodes[s]; dx,dy,_=nodes[d]
    ax.annotate('',xy=(dx,dy),xytext=(sx,sy),
        arrowprops=dict(arrowstyle='->',lw=2.5,color=col,shrinkA=33,shrinkB=33))
    ax.text((sx+dx)/2,(sy+dy)/2+0.38,lbl,ha='center',fontsize=12,color=col,fontweight='bold')
ax.text(5,0.4,'Backdoor: X←C→Y  (контролируем C для несмещённой θ)',
        ha='center',fontsize=10,color='red',style='italic')
plt.tight_layout(); plt.show()

---
## STEP 2 — Variable Selection

In [ ]:
def generate_s1(T=600, t0=300, seed=42):
    rng = np.random.RandomState(seed)
    C=np.zeros(T); X=np.zeros(T); Y=np.zeros(T)
    ec,ex,ey = rng.randn(T), rng.randn(T), rng.randn(T)
    for t in range(1,T):
        C[t] = 0.6*C[t-1] + ec[t]
        X[t] = 0.3*X[t-1] + 0.4*C[t-1] + ex[t]
        Y[t] = 0.2*Y[t-1] + 0.8*X[t-1] - 0.3*C[t-1] + ey[t]
    delta = 2.0 * np.std(X)
    Y_int=np.zeros(T); X_int=X.copy()
    X_int[t0] += delta
    Y_int[:t0+1] = Y[:t0+1]
    for t in range(t0+1, T):
        Y_int[t] = 0.2*Y_int[t-1] + 0.8*X_int[t-1] - 0.3*C[t-1] + ey[t]
    df    = pd.DataFrame({'Y':Y_int,'X':X_int,'C':C})
    df_cf = pd.DataFrame({'Y':Y,    'X':X,    'C':C})
    return df, df_cf, delta

df, df_cf, DELTA = generate_s1(T=T, t0=T0, seed=SEED)
print(f'Данные: T={T}, t0={T0}, Δ={DELTA:.4f}')
df.describe().round(3)

In [ ]:
print('ADF тест:')
for col in ['Y','X','C']:
    s,p,*_ = adfuller(df[col]); print(f'  {col}: stat={s:.3f}  p={p:.4f}  {"✓ I(0)" if p<0.05 else "✗ нестац."}')

In [ ]:
# Экспертный (backdoor criterion)
controls_expert = [v for v in DAG.confounders if v not in DAG.mediators]

# Эвристический (корреляция)
def select_heuristic(df, target, candidates, top_k=5):
    y=df[target]
    scores={c:(abs(df[c].corr(y,'pearson'))+abs(df[c].corr(y,'spearman')))/2 for c in candidates}
    return [k for k,_ in sorted(scores.items(),key=lambda x:-x[1])[:top_k]]

controls_heuristic = select_heuristic(df,'Y',['X','C'],top_k=2)
print('Экспертный (backdoor)  :', controls_expert)
print('Эвристический (корр.)  :', controls_heuristic)

# Визуализация
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('STEP 2 — Анализ переменных',fontweight='bold')
corr=df.corr()
im=axes[0].imshow(corr,cmap='RdBu_r',vmin=-1,vmax=1)
plt.colorbar(im,ax=axes[0],shrink=0.8)
for i in range(3):
    for j in range(3):
        axes[0].text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=12)
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(corr.columns); axes[0].set_yticklabels(corr.columns)
axes[0].set_title('Корреляции')
axes[1].scatter(df['X'],df['Y'],alpha=0.2,s=8,c='steelblue')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].set_title(f'X↔Y: ρ={corr.loc["X","Y"]:.3f}'); axes[1].grid(alpha=0.3)
axes[2].scatter(df['C'],df['Y'],alpha=0.2,s=8,c='darkorange')
axes[2].set_xlabel('C'); axes[2].set_ylabel('Y')
axes[2].set_title(f'C↔Y: ρ={corr.loc["C","Y"]:.3f}'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## STEP 3 — Causal Models

1. **OLS + backdoor** — полная правильная спецификация с конфаундером
2. **DML** — partialling-out с 5-fold cross-fitting
3. **VAR(p) + Granger** — авторегрессионная система с тестом Гренджера

In [ ]:
def make_reg_matrix(df, target, controls, lags=1):
    all_vars=[target]+controls
    feat={f'{v}_lag{lag}':df[v].shift(lag) for lag in range(1,lags+1) for v in all_vars}
    fd=pd.DataFrame(feat).dropna()
    y=df[target].loc[fd.index]
    return fd, y

def metrics(yt, yp, name):
    mse=mean_squared_error(yt,yp)
    return {'Модель':name,'MSE':round(mse,4),'RMSE':round(np.sqrt(mse),4),'MAE':round(mean_absolute_error(yt,yp),4)}

feat_df, y_ser = make_reg_matrix(df,'Y',['X','C'],lags=1)
feat_cols = list(feat_df.columns)
X_all = feat_df.values; y_all = y_ser.values
te = TRAIN_END - 1
X_tr,y_tr = X_all[:te],y_all[:te]
X_te,y_te = X_all[te:],y_all[te:]
print('Признаки:',feat_cols)
print(f'Train={len(y_tr)}  Test={len(y_te)}')

In [ ]:
# ── 3.1: OLS + backdoor ──────────────────────────────────────────────────
ols_full = OLS(y_tr, add_constant(X_tr)).fit()
x_lag1_idx = feat_cols.index('X_lag1')
theta_ols = ols_full.params[x_lag1_idx+1]
preds_ols = ols_full.predict(add_constant(X_te))
m_ols = metrics(y_te, preds_ols, 'OLS+backdoor')

print('OLS + backdoor:')
print(f'  Коэффициенты: {dict(zip(["const"]+feat_cols, np.round(ols_full.params,4)))}')
print(f'  θ̂(X_lag1)   = {theta_ols:.4f}  (θ*={TRUE_EFFECT})')
print(f'  Метрики     : {m_ols}')

In [ ]:
# ── 3.2: DML (Double/Debiased ML, partialling-out) ────────────────────────
def run_dml(X_tr,y_tr,X_te,y_te,cause_col,control_cols,feat_cols,n_folds=5):
    d_idx=feat_cols.index(cause_col)
    w_idx=[feat_cols.index(c) for c in control_cols]
    D=X_tr[:,d_idx]; W=X_tr[:,w_idx]; n=len(y_tr)
    Y_res=np.zeros(n); D_res=np.zeros(n)
    fsize=n//n_folds
    for k in range(n_folds):
        val=np.arange(k*fsize,min((k+1)*fsize,n))
        tr_k=np.concatenate([np.arange(0,k*fsize),np.arange(min((k+1)*fsize,n),n)])
        if len(tr_k)<10: continue
        my=Ridge(1.0).fit(W[tr_k],y_tr[tr_k]); md_=Ridge(1.0).fit(W[tr_k],D[tr_k])
        Y_res[val]=y_tr[val]-my.predict(W[val])
        D_res[val]=D[val]-md_.predict(W[val])
    theta=np.dot(D_res,Y_res)/(np.dot(D_res,D_res)+1e-12)
    resid=Y_res-theta*D_res
    se=np.sqrt(np.mean(resid**2)/(np.dot(D_res,D_res)+1e-12))
    ci=(theta-1.96*se, theta+1.96*se)
    m_pred=Ridge(0.5).fit(X_tr,y_tr)
    preds=m_pred.predict(X_te)
    return {'theta_hat':theta,'se':se,'ci':ci,'preds':preds,'metrics':metrics(y_te,preds,'DML')}

res_dml=run_dml(X_tr,y_tr,X_te,y_te,'X_lag1',['Y_lag1','C_lag1'],feat_cols)
print('DML (5-fold cross-fitting, partialling-out):')
print(f'  θ̂  = {res_dml["theta_hat"]:.4f}  (θ*={TRUE_EFFECT})')
print(f'  SE  = {res_dml["se"]:.4f}')
print(f'  95% CI = [{res_dml["ci"][0]:.4f}, {res_dml["ci"][1]:.4f}]')
print(f'  θ* в CI: {"✓" if res_dml["ci"][0]<=TRUE_EFFECT<=res_dml["ci"][1] else "✗"}')
print(f'  Метрики: {res_dml["metrics"]}')

In [ ]:
# ── 3.3: VAR(p) + Granger ────────────────────────────────────────────────
def run_var_granger(df,target,controls,train_end):
    all_vars=[target]+controls
    df_var=df[all_vars].iloc[:train_end]
    try:    best_lag=max(1,VAR(df_var).fit(maxlags=4,ic='aic').k_ar)
    except: best_lag=2
    var_fit=VAR(df_var).fit(best_lag)
    x_pos=all_vars.index('X')
    theta_v=var_fit.coefs[0,0,x_pos]

    granger_pvals={}
    for cause in controls:
        try:
            gr=grangercausalitytests(df[[target,cause]].iloc[:train_end],maxlag=best_lag,verbose=False)
            granger_pvals[cause]=round(gr[best_lag][0]['ssr_ftest'][1],4)
        except: granger_pvals[cause]=1.0

    # Rolling forecast
    preds=[]
    for i in range(train_end,T):
        win=df[all_vars].iloc[max(0,i-200):i]
        try:
            m=VAR(win).fit(best_lag)
            preds.append(m.forecast(win.values[-best_lag:],steps=1)[0,0])
        except: preds.append(df[target].iloc[i-1])

    y_te_v=df[target].iloc[train_end:].values[:len(preds)]
    return {'theta_hat':theta_v,'granger_pvals':granger_pvals,'best_lag':best_lag,
            'preds':np.array(preds),'y_test':y_te_v,
            'metrics':metrics(y_te_v,np.array(preds),'VAR+Granger')}

print('Запуск VAR+Granger (rolling)...')
res_var=run_var_granger(df,'Y',['X','C'],TRAIN_END)
print(f'  Лаг AIC           : {res_var["best_lag"]}')
print(f'  Granger p-values  : {res_var["granger_pvals"]}')
print(f'  θ̂(X_lag1) из VAR : {res_var["theta_hat"]:.4f}  (θ*={TRUE_EFFECT})')
print(f'  Метрики           : {res_var["metrics"]}')

---
## STEP 4 — Baseline ML Models

1. **ARIMA(2,0,1)** — только Y, без X и C
2. **Ridge (без C)** — смещённая оценка θ из-за отсутствия конфаундера
3. **RandomForest** — 300 деревьев, все лаговые признаки
4. **LightGBM** — градиентный бустинг

In [ ]:
# ── 4.1: ARIMA ───────────────────────────────────────────────────────────
print('ARIMA rolling...')
y_full=df['Y'].values
preds_arima=[]
for i in range(TRAIN_END,T):
    try:    preds_arima.append(ARIMA(y_full[:i],order=(2,0,1)).fit().forecast(1)[0])
    except: preds_arima.append(y_full[i-1])
y_te_arima=y_full[TRAIN_END:]
res_arima={'preds':np.array(preds_arima),'y_test':y_te_arima,
           'metrics':metrics(y_te_arima,np.array(preds_arima),'ARIMA'),'theta_hat':np.nan}
print(f'  {res_arima["metrics"]}')

In [ ]:
# ── 4.2: Ridge без C (наивный) ────────────────────────────────────────────
fd_n,y_n=make_reg_matrix(df,'Y',['X'],lags=1)
Xn=fd_n.values; yn=y_n.values; te_n=TRAIN_END-1
rn=Ridge(0.5).fit(Xn[:te_n],yn[:te_n])
theta_naive=rn.coef_[list(fd_n.columns).index('X_lag1')]
preds_naive=rn.predict(Xn[te_n:]); yte_naive=yn[te_n:]
res_naive={'preds':preds_naive,'y_test':yte_naive,
           'metrics':metrics(yte_naive,preds_naive,'Ridge (без C)'),'theta_hat':theta_naive}
print(f'Ridge (без C):  θ̂={theta_naive:.4f}  смещение={theta_naive-TRUE_EFFECT:+.4f}')
print(f'  {res_naive["metrics"]}')

In [ ]:
# ── 4.3: RandomForest ────────────────────────────────────────────────────
rf=RandomForestRegressor(n_estimators=300,random_state=SEED,n_jobs=-1)
rf.fit(X_tr,y_tr)
preds_rf=rf.predict(X_te)
res_rf={'preds':preds_rf,'y_test':y_te,'metrics':metrics(y_te,preds_rf,'RandomForest'),'theta_hat':np.nan}
print('RandomForest FI:',dict(zip(feat_cols,np.round(rf.feature_importances_,3))))
print(f'  {res_rf["metrics"]}')

In [ ]:
# ── 4.4: LightGBM ────────────────────────────────────────────────────────
lgbm=lgb.LGBMRegressor(n_estimators=300,learning_rate=0.05,num_leaves=31,verbosity=-1,random_state=SEED)
lgbm.fit(X_tr,y_tr)
preds_lgb=lgbm.predict(X_te)
res_lgb={'preds':preds_lgb,'y_test':y_te,'metrics':metrics(y_te,preds_lgb,'LightGBM'),'theta_hat':np.nan}
print('LightGBM FI:',dict(zip(feat_cols,np.round(lgbm.feature_importances_/lgbm.feature_importances_.sum(),3))))
print(f'  {res_lgb["metrics"]}')

---
## STEP 5 — Comparison Report

In [ ]:
ALL_RESULTS=[
    ('OLS+backdoor','Каузальная',theta_ols,   preds_ols,       y_te,            m_ols),
    ('DML',         'Каузальная',res_dml['theta_hat'],res_dml['preds'],y_te,    res_dml['metrics']),
    ('VAR+Granger', 'Каузальная',res_var['theta_hat'],res_var['preds'],res_var['y_test'],res_var['metrics']),
    ('ARIMA',       'ML-базлайн',np.nan,      res_arima['preds'],res_arima['y_test'],res_arima['metrics']),
    ('Ridge (без C)','ML-базлайн',theta_naive,preds_naive,     yte_naive,       res_naive['metrics']),
    ('RandomForest','ML-базлайн',np.nan,      preds_rf,        y_te,            res_rf['metrics']),
    ('LightGBM',    'ML-базлайн',np.nan,      preds_lgb,       y_te,            res_lgb['metrics']),
]

rows=[]
for name,cat,theta,preds,ytrue,m in ALL_RESULTS:
    bias=(theta-TRUE_EFFECT) if not np.isnan(theta) else np.nan
    rows.append({'Модель':name,'Категория':cat,'MSE':m['MSE'],'RMSE':m['RMSE'],'MAE':m['MAE'],
                 'θ̂':round(theta,4) if not np.isnan(theta) else '—',
                 'Bias':round(bias,4) if not np.isnan(bias) else '—'})

report=pd.DataFrame(rows).set_index('Модель')
print(f'θ* = {TRUE_EFFECT}\n')
report

In [ ]:
# ── 5.1: Grouped bar chart MSE / RMSE ────────────────────────────────────
names=report.index.tolist()
cats_r=report['Категория'].tolist()
colors=['#1f77b4' if c=='Каузальная' else '#ff7f0e' for c in cats_r]

fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle('STEP 5.1 — MSE и RMSE',fontsize=13,fontweight='bold')
for ax,mn in zip(axes,['MSE','RMSE']):
    vals=report[mn].tolist()
    bars=ax.bar(names,vals,color=colors,edgecolor='black',lw=0.8)
    ax.set_title(mn); ax.set_ylabel(mn)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names,rotation=30,ha='right',fontsize=10)
    ax.grid(axis='y',alpha=0.4)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,v+0.005,f'{v:.3f}',ha='center',va='bottom',fontsize=9)
cp=mpatches.Patch(color='#1f77b4',label='Каузальные')
mp=mpatches.Patch(color='#ff7f0e',label='ML-базлайны')
axes[0].legend(handles=[cp,mp])
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.2: Восстановление θ* ───────────────────────────────────────────────
theta_data=[(n,t,t-TRUE_EFFECT) for n,c,t,*_ in ALL_RESULTS if not np.isnan(t)]
tnames_=[r[0] for r in theta_data]
thetas_=[r[1] for r in theta_data]
biases_=[r[2] for r in theta_data]
tcolors=['#1f77b4' if n in ('OLS+backdoor','DML','VAR+Granger') else '#ff7f0e' for n in tnames_]

fig,axes=plt.subplots(1,2,figsize=(12,5))
fig.suptitle('STEP 5.2 — Восстановление θ* = 0.8',fontsize=13,fontweight='bold')

axes[0].bar(tnames_,thetas_,color=tcolors,edgecolor='black')
axes[0].axhline(TRUE_EFFECT,color='red',lw=2,ls='--',label=f'θ*={TRUE_EFFECT}')
axes[0].set_title('Оценки θ̂'); axes[0].set_ylabel('θ̂')
axes[0].set_xticks(range(len(tnames_))); axes[0].set_xticklabels(tnames_,rotation=20,ha='right')
axes[0].legend(); axes[0].grid(axis='y',alpha=0.4)
for i,v in enumerate(thetas_): axes[0].text(i,v+0.01,f'{v:.3f}',ha='center',fontsize=10)

bcols=['green' if abs(b)<0.05 else 'orange' if abs(b)<0.2 else 'red' for b in biases_]
axes[1].bar(tnames_,biases_,color=bcols,edgecolor='black')
axes[1].axhline(0,color='black',lw=1.5)
axes[1].set_title('Смещение θ̂ − θ*'); axes[1].set_ylabel('Bias')
axes[1].set_xticks(range(len(tnames_))); axes[1].set_xticklabels(tnames_,rotation=20,ha='right')
axes[1].grid(axis='y',alpha=0.4)
for i,v in enumerate(biases_): axes[1].text(i,v+(0.005 if v>=0 else -0.02),f'{v:+.3f}',ha='center',fontsize=10)

plt.tight_layout(); plt.show()

In [ ]:
# ── 5.3: Прогнозы на тестовом периоде ────────────────────────────────────
test_t=np.arange(TRAIN_END,T)
fig,axes=plt.subplots(2,1,figsize=(14,9),sharex=True)
fig.suptitle(f'STEP 5.3 — Прогнозы [t={TRAIN_END}..{T}]',fontsize=13,fontweight='bold')
y_fact=df['Y'].values[TRAIN_END:]

for ax in axes:
    ax.plot(test_t,y_fact,'black',lw=2.2,label='Y (факт)',zorder=10)
    ax.axvline(T0,color='purple',ls=':',lw=1.5,label=f't₀={T0} (вмешательство)')
    ax.grid(alpha=0.3)

causal_items=[(n,p,m,'#1f77b4') for n,c,_,p,_,m in ALL_RESULTS[:1]] +              [(n,p,m,'#2ca02c') for n,c,_,p,_,m in ALL_RESULTS[1:2]] +              [(n,p,m,'#9467bd') for n,c,_,p,yt,m in ALL_RESULTS[2:3]]
ml_items=    [(n,p,m,'#d62728') for n,c,_,p,yt,m in ALL_RESULTS[3:4]] +              [(n,p,m,'#ff7f0e') for n,c,_,p,yt,m in ALL_RESULTS[4:5]] +              [(n,p,m,'#8c564b') for n,c,_,p,yt,m in ALL_RESULTS[5:6]] +              [(n,p,m,'#e377c2') for n,c,_,p,yt,m in ALL_RESULTS[6:7]]

for n,p,m,col in causal_items:
    nn=min(len(p),len(test_t))
    axes[0].plot(test_t[:nn],p[:nn],color=col,lw=1.5,alpha=0.9,label=f'{n} (MSE={m["MSE"]})')
for n,p,m,col in ml_items:
    nn=min(len(p),len(test_t))
    axes[1].plot(test_t[:nn],p[:nn],color=col,lw=1.5,alpha=0.9,label=f'{n} (MSE={m["MSE"]})')

axes[0].set_title('Каузальные модели'); axes[0].legend(fontsize=10,loc='upper right')
axes[1].set_title('ML-базлайны');       axes[1].legend(fontsize=10,loc='upper right')
axes[1].set_xlabel('t')
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.4: Heatmap нормированных метрик ────────────────────────────────────
heat_df=report[['MSE','RMSE','MAE']].copy()
heat_norm=(heat_df-heat_df.min())/(heat_df.max()-heat_df.min()+1e-9)

fig,ax=plt.subplots(figsize=(7,5.5))
im=ax.imshow(heat_norm.values,cmap='RdYlGn_r',vmin=0,vmax=1,aspect='auto')
plt.colorbar(im,ax=ax,label='0=лучший, 1=худший')
ax.set_xticks(range(3)); ax.set_xticklabels(['MSE','RMSE','MAE'],fontsize=12)
ax.set_yticks(range(len(heat_df))); ax.set_yticklabels(heat_df.index,fontsize=10)
ax.set_title('Heatmap метрик (нормировка min-max)',fontweight='bold')
for i in range(len(heat_df)):
    for j in range(3):
        ax.text(j,i,f'{heat_df.iloc[i,j]:.3f}',ha='center',va='center',fontsize=9)
ax.axhline(2.5,color='white',lw=3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.5: Monte Carlo (10 реализаций) — Bias / RMSE_θ / Coverage 95% ─────
print('Monte Carlo...')
mc=[]
for i in range(N_REAL):
    df_i,_,_=generate_s1(T=T,t0=T0,seed=SEED+i)
    fd_i,y_i=make_reg_matrix(df_i,'Y',['X','C'],lags=1)
    Xf_i=fd_i.values; yi_arr=y_i.values; te_i=TRAIN_END-1
    X_tr_i,y_tr_i=Xf_i[:te_i],yi_arr[:te_i]
    X_te_i,y_te_i=Xf_i[te_i:],yi_arr[te_i:]
    feat_i=list(fd_i.columns)

    ols_i=OLS(y_tr_i,add_constant(X_tr_i)).fit()
    xi=feat_i.index('X_lag1')
    theta_ols_i=ols_i.params[xi+1]

    dml_i=run_dml(X_tr_i,y_tr_i,X_te_i,y_te_i,'X_lag1',['Y_lag1','C_lag1'],feat_i)
    in_ci_i=int(dml_i['ci'][0]<=TRUE_EFFECT<=dml_i['ci'][1])

    fd_n_i,y_n_i=make_reg_matrix(df_i,'Y',['X'],lags=1)
    rn_i=Ridge(0.5).fit(fd_n_i.values[:te_i],y_n_i.values[:te_i])
    theta_naive_i=rn_i.coef_[list(fd_n_i.columns).index('X_lag1')]

    mc.append({'theta_ols':theta_ols_i,'theta_dml':dml_i['theta_hat'],
               'theta_naive':theta_naive_i,'in_ci':in_ci_i})

mc_df=pd.DataFrame(mc)
summary_mc=pd.DataFrame({
    'Метод':['OLS+backdoor','DML','Ridge (без C)'],
    'Bias':[round(mc_df['theta_ols'].mean()-TRUE_EFFECT,4),
            round(mc_df['theta_dml'].mean()-TRUE_EFFECT,4),
            round(mc_df['theta_naive'].mean()-TRUE_EFFECT,4)],
    'RMSE(θ)':[round(np.sqrt(((mc_df['theta_ols']-TRUE_EFFECT)**2).mean()),4),
               round(np.sqrt(((mc_df['theta_dml']-TRUE_EFFECT)**2).mean()),4),
               round(np.sqrt(((mc_df['theta_naive']-TRUE_EFFECT)**2).mean()),4)],
    'Coverage 95%':['—',f'{mc_df["in_ci"].mean()*100:.0f}%','—']
}).set_index('Метод')
print(f'θ* = {TRUE_EFFECT}  |  {N_REAL} реализаций\n')
summary_mc

In [ ]:
# ── 5.6: Распределение θ̂ по MC реализациям ──────────────────────────────
fig,ax=plt.subplots(figsize=(10,5))
bins=np.linspace(0.4,1.2,25)
ax.hist(mc_df['theta_ols'],  bins=bins,alpha=0.6,color='#1f77b4',label='OLS+backdoor')
ax.hist(mc_df['theta_dml'],  bins=bins,alpha=0.6,color='#2ca02c',label='DML')
ax.hist(mc_df['theta_naive'],bins=bins,alpha=0.6,color='#d62728',label='Ridge (без C)')
ax.axvline(TRUE_EFFECT,color='black',lw=2.5,ls='--',label=f'θ*={TRUE_EFFECT}')
ax.set_title(f'Распределение θ̂ по {N_REAL} реализациям')
ax.set_xlabel('θ̂'); ax.set_ylabel('Частота')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.7: Финальный отчёт ─────────────────────────────────────────────────
bc_nm=report.loc[['OLS+backdoor','DML','VAR+Granger'],'MSE'].idxmin()
bm_nm=report.loc[['ARIMA','Ridge (без C)','RandomForest','LightGBM'],'MSE'].idxmin()
mse_bc=report.loc[bc_nm,'MSE']; mse_bm=report.loc[bm_nm,'MSE']
delta_pct=(mse_bm-mse_bc)/mse_bm*100

print('══════════════════════════════════════════════')
print('          ИТОГОВЫЙ ОТЧЁТ')
print('══════════════════════════════════════════════')
print(f'  Истинный θ*          = {TRUE_EFFECT}')
print(f'  Лучший каузальный    : {bc_nm:20s}  MSE={mse_bc:.4f}')
print(f'  Лучший ML-базлайн    : {bm_nm:20s}  MSE={mse_bm:.4f}')
s='+' if delta_pct>0 else ''
print(f'  Разница MSE          : {s}{delta_pct:.1f}%  ({"каузальные лучше" if delta_pct>0 else "ML-базлайны лучше"})')
print()
print('  Восстановление θ (одна реализация):')
for name,cat,theta,preds,ytrue,m in ALL_RESULTS:
    if not np.isnan(theta):
        print(f'    {name:20s}  θ̂={theta:.4f}  Bias={theta-TRUE_EFFECT:+.4f}')
print()
print('  Monte Carlo:')
for idx,row in summary_mc.iterrows():
    print(f'    {str(idx):20s}  Bias={row["Bias"]:+.4f}  RMSE(θ)={row["RMSE(θ)"]:.4f}  Coverage={row["Coverage 95%"]}')
print()
print('  ВЫВОД:')
print('    Минимизация MSE ≠ несмещённость θ.')
print('    Контроль конфаундера C необходим для корректной оценки θ(X→Y).')
print('    ML-модели без C дают систематически смещённую θ̂.')

In [ ]:
# ── 5.8: Итоговая таблица с подсветкой ───────────────────────────────────
def highlight(col):
    try:
        num=pd.to_numeric(col,errors='coerce')
        s=['']*len(col)
        if num.notna().sum()>1:
            s[int(num.idxmin())]='background-color:#c6efce'
            s[int(num.idxmax())]='background-color:#ffc7ce'
        return s
    except: return ['']*len(col)

(report.drop(columns=['Bias'],errors='ignore')
       .style
       .apply(highlight,subset=['MSE','RMSE','MAE'])
       .set_caption(f'Итоговая таблица: θ* = {TRUE_EFFECT}. 🟢 лучший, 🔴 худший.'))

---
## Заключение

| Критерий | Наблюдение |
|----------|------------|
| **MSE / RMSE** | Все модели сопоставимы; ML-базлайны не уступают каузальным по точности прогноза |
| **Восстановление θ*** | OLS+backdoor и VAR близки к 0.8; Ridge без C **систематически смещён** |
| **DML Coverage 95%** | Доверительные интервалы DML покрывают θ* в большинстве реализаций |
| **Monte Carlo Bias** | OLS+backdoor — минимальный Bias; Ridge (без C) — максимальный |

> **Главный вывод:** минимизация MSE не требует каузального подхода, но **корректная оценка причинного эффекта** при наличии конфаундера возможна только с контролем backdoor-пути.